# Dim Color — Gold Layer

Enriched color dimension with a derived `color_family` grouping based on the RGB hex value.

## What this notebook does

1. **Read** — Loads all records from the silver `foundation_colors` table.
2. **Transform** — Converts RGB hex to component values and classifies each color into a `color_family` (Red, Blue, Green, Yellow, Orange, Purple, Black, White, Gray).
3. **Write** — Overwrites the Delta table at `/Volumes/lego_brickbase/gold/delta_volume/dim_color`.
4. **Register** — Creates the catalog table `lego_brickbase.gold.dim_color`.

## Imports and Configuration

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType


SILVER_TABLE     = "lego_brickbase.silver.foundation_colors"
DELTA_TABLE_PATH = "/Volumes/lego_brickbase/gold/delta_volume/dim_color"
CATALOG_TABLE    = "lego_brickbase.gold.dim_color"

## Read and Transform

Parse the RGB hex string into red, green, blue components and classify into a color family based on dominant channel and brightness thresholds.

In [ ]:
def classify_color_family(rgb_hex):
    """Classify a 6-digit hex RGB string into a color family."""
    if rgb_hex is None or len(rgb_hex) != 6:
        return "Unknown"
    try:
        r = int(rgb_hex[0:2], 16)
        g = int(rgb_hex[2:4], 16)
        b = int(rgb_hex[4:6], 16)
    except ValueError:
        return "Unknown"

    brightness = (r + g + b) / 3.0

    if brightness < 30:
        return "Black"
    if brightness > 230 and max(r, g, b) - min(r, g, b) < 30:
        return "White"
    if max(r, g, b) - min(r, g, b) < 30:
        return "Gray"

    if r > g and r > b:
        if g > 150 and b < 100:
            return "Yellow"
        if g > 80 and g < 160 and b < 80:
            return "Orange"
        if b > g:
            return "Purple"
        return "Red"
    if g > r and g > b:
        if r > 180 and b < 100:
            return "Yellow"
        return "Green"
    if b > r and b > g:
        if r > 100:
            return "Purple"
        return "Blue"
    return "Other"

classify_color_family_udf = udf(classify_color_family, StringType())

df_colors = spark.table(SILVER_TABLE)

df_dim = df_colors.select(
    col("color_key"),
    col("color_id"),
    col("color_name"),
    col("rgb"),
    col("is_transperent"),
    classify_color_family_udf(col("rgb")).alias("color_family"),
)

df_dim.printSchema()
df_dim.display(10, truncate=False)

## Write to Delta Volume, Register Catalog Table, and Apply Constraints

In [ ]:
(
    df_dim
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DELTA_TABLE_PATH)
)

row_count = spark.read.format("delta").load(DELTA_TABLE_PATH).count()
print(f"Rows written to Delta table: {row_count:,}")

spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG_TABLE} (
        color_key      INTEGER NOT NULL COMMENT 'Surrogate key for the color dimension',
        color_id       INTEGER          COMMENT 'Original color identifier from the Rebrickable source system',
        color_name     STRING           COMMENT 'Human-readable name of the LEGO color (e.g. Bright Red, Dark Azure)',
        rgb            STRING           COMMENT 'Six-digit hexadecimal RGB value representing the color (e.g. FF0000)',
        is_transperent BOOLEAN          COMMENT 'True if the color is a transparent variant',
        color_family   STRING           COMMENT 'Derived broad color grouping based on RGB channel analysis (Red, Blue, Green, Yellow, Orange, Purple, Black, White, Gray, Other)',
        CONSTRAINT dim_color_pk PRIMARY KEY (color_key)
    )
    COMMENT 'Color dimension enriched from silver foundation_colors. Adds a derived color_family grouping based on RGB hex analysis.'
""")

spark.sql(f"""
    INSERT INTO {CATALOG_TABLE}
    SELECT * FROM delta.`{DELTA_TABLE_PATH}`
""")

print(f"Catalog table ready with constraints: {CATALOG_TABLE}")